## 12. Análise lexical e de conteúdo (complementa os embeddings)

Até aqui medimos a proximidade entre países por **embeddings e clusters**. Os clusters seguem o idioma (NMI alto), então esta seção olha o **vocabulário e o conteúdo** de outras formas, todas com a mesma pergunta: *os países estão ficando mais parecidos?* Cada método responde a uma parte da teoria:

| Método | O que mede | Por que ajuda na teoria |
|---|---|---|
| 12.2 Palavras frequentes e distintivas (log-odds, *Fightin' Words*) | o que cada cluster/país/ano diz de característico | mostra *o quê* separa os países, não só *quanto* |
| 12.3 Similaridade lexical (JS e cosseno TF-IDF sobre palavras) | distância de vocabulário entre países, por ano | testa H1/H2 sem depender de clusters nem de k |
| 12.4 Vizinhos semânticos e frases parecidas | pares de músicas de países diferentes que dizem coisas parecidas | dá exemplos concretos da proximidade |
| 12.5 Frases (trigramas) compartilhadas | fórmulas verbais comuns a vários países | indício de circulação de um repertório global |
| 12.6 Homofilia (vizinhança) | se músicas têm vizinhos do mesmo país mais do que o acaso | *homophily* (McPherson et al., 2001): queda = mistura |
| 12.7 Previsibilidade do país (classificador) | o quanto o texto "denuncia" o país | teste de duas amostras por classificador (Lopez-Paz e Oquab, 2017) |
| 12.8 Informação mútua país–cluster e diversidade interna | quanto do cluster é explicado pelo país | queda = homogeneização entre países |
| 12.9 Tópicos (LDA) | convergência temática, independente de k-means | análise cultural por tópicos (DiMaggio et al., 2013) |
| 12.10 Sentimento e complexidade | tom emocional, repetição e compressibilidade | convergência de *estilo*, não só de tema |

**Cuidados:** (i) o texto comum é a tradução para o inglês (`FONTE_TEXTO`), logo os resultados herdam os erros da tradução automática; (ii) com ~40 músicas por célula, medidas sobre palavras têm viés de amostra, por isso comparamos sempre com o **nulo por permutação** do país; (iii) tudo é correlacional.

In [ ]:
# 12.0 — preparação
import gzip
import scipy.sparse as sp
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from scipy.stats import entropy

FONTE_TEXTO = "letra_en"        # texto comum em inglês; "letra_limpa" usa o idioma original (mistura línguas)
N_BOOT_LEX  = min(N_BOOT, 300)
N_PERM_LEX  = min(N_PERM, 200)
N_TOPICOS   = 8
K_VIZ       = 10                # vizinhos usados na homofilia
PASTA_LEX   = PASTA_RES / "lexico"
PASTA_LEX.mkdir(exist_ok=True)
COR_PAIS = dict(zip(PAISES, ["#2a78d6", "#1baf7a", "#eb6834", "#8a63d2", "#c9a227", "#52514e"]))

_EXTRA = ("que de la el en y a los las se no me te lo le mi tu un una por con para es del al como mas pero yo si ya "
          "eso esta este todo nada o e do da os as um uma com pra não nao eu você voce ele ela meu minha seu sua "
          "ta tá tô vai vou oh ah eh yeah yeh ey hey ooh uh na woah whoa ayy ay nah mmm hmm em ni dont im youre "
          "cause got gonna wanna let just like know ive ill thats cant wont aint didnt isnt wasnt youll theyre "
          "shes hes whats lets gotta cuz").split()
STOP = set(ENGLISH_STOP_WORDS) | set(_EXTRA)
_prep = lambda s: re.sub(r"[’'`]", "", str(s).lower())

fx = faixas[faixas[FONTE_TEXTO].fillna("").astype(str).str.strip() != ""].copy()
LPOS = {t: i for i, t in enumerate(fx.index)}
d_lex = dados[dados["track_id"].isin(fx.index)].copy()
d_lex["ix"] = d_lex["track_id"].map(LPOS)
ANOS = np.array(sorted(d_lex["year"].unique()))
print(f"{len(fx)} de {len(faixas)} faixas com texto em '{FONTE_TEXTO}' | {len(d_lex)} linhas país × ano")

cv = CountVectorizer(stop_words=list(STOP), preprocessor=_prep, min_df=5, token_pattern=r"(?u)\b[a-zA-Z]{3,}\b")
X = cv.fit_transform(fx[FONTE_TEXTO]).tocsr().astype(np.float64)
VOC = np.array(cv.get_feature_names_out())
IDF = np.log((1 + X.shape[0]) / (1 + np.asarray((X > 0).sum(0)).ravel())) + 1
print(f"vocabulário: {len(VOC)} palavras | {int(X.sum())} ocorrências")

def conta(mask):
    # soma das contagens de palavras das linhas (país × ano) selecionadas
    return np.asarray(X[d_lex.loc[mask, "ix"].to_numpy()].sum(0)).ravel()

## 12.2 Palavras frequentes e distintivas

Frequência bruta mostra o que é comum a todos (amor, noite, baby). Para entender o que **separa** os grupos usamos a razão de chances logarítmica com prior informativo (*Fightin' Words*, Monroe, Colaresi e Quinn, 2008): para cada palavra, compara a frequência no grupo com a do resto e devolve um escore *z*. Vale como: palavras com *z* alto são características do grupo, e o prior evita que palavras raras dominem a lista.

In [ ]:
# 12.2 — palavras frequentes e distintivas
def fightin(yi, yj, a0=500.0):
    # log-odds com prior de Dirichlet informativo (prior = frequência conjunta); devolve z (positivo = distintiva de i)
    bg = yi + yj + 1.0
    a = a0 * bg / bg.sum()
    ni, nj = yi.sum(), yj.sum()
    li = np.log((yi + a) / (ni + a0 - yi - a))
    lj = np.log((yj + a) / (nj + a0 - yj - a))
    return (li - lj) / np.sqrt(1 / (yi + a) + 1 / (yj + a))

def top_termos(m_grupo, m_resto, n=12, min_cont=10):
    yi, yj = conta(m_grupo), conta(m_resto)
    z = fightin(yi, yj)
    z = np.where(yi + yj >= min_cont, z, -np.inf)
    o = np.argsort(-z)[:n]
    return [(VOC[k], float(z[k]), 1000 * yi[k] / max(yi.sum(), 1)) for k in o if np.isfinite(z[k])]

# (a) por cluster
lin = []
for c in range(K):
    m = d_lex["cluster"] == c
    if m.sum() == 0:
        continue
    for r, (w, z, f) in enumerate(top_termos(m, ~m), 1):
        lin.append({"cluster": c, "rank": r, "termo": w, "z": z, "freq_por_mil": f})
lex_cluster = pd.DataFrame(lin)
print("Termos distintivos por cluster:")
display(lex_cluster.groupby("cluster")["termo"].apply(", ".join).to_frame("termos"))

# (b) por país (todos os anos), distintivas e frequências brutas
lin, brutas = [], {}
for cc in PAISES:
    m = d_lex["cc"] == cc
    for r, (w, z, f) in enumerate(top_termos(m, ~m), 1):
        lin.append({"cc": cc, "rank": r, "termo": w, "z": z, "freq_por_mil": f})
    y = conta(m)
    brutas[NOMES[cc]] = ", ".join(f"{VOC[k]} ({1000 * y[k] / y.sum():.0f})" for k in np.argsort(-y)[:10])
lex_pais = pd.DataFrame(lin)
print("\nPalavras mais frequentes por país (entre parênteses: ocorrências por mil palavras):")
with pd.option_context("display.max_colwidth", 140):
    display(pd.Series(brutas, name="mais frequentes").to_frame())
print("Palavras distintivas por país (vs. os demais):")
display(lex_pais.groupby("cc")["termo"].apply(", ".join).rename(index=NOMES).to_frame("termos"))

fig, axes = plt.subplots(2, 3, figsize=(11, 6.2))
for ax, cc in zip(axes.flat, PAISES):
    g = lex_pais[lex_pais["cc"] == cc].head(10).iloc[::-1]
    ax.barh(g["termo"], g["z"], color=COR_PAIS[cc])
    ax.set_title(NOMES[cc], loc="left"); ax.grid(axis="y", visible=False)
fig.suptitle("Palavras distintivas de cada país (escore z do log-odds)", x=0.01, ha="left", fontweight="bold")
fig.tight_layout(); fig.savefig(PASTA_LEX / "palavras_distintivas_pais.png"); plt.show()

# (c) por país e ano: 5 palavras distintivas de cada célula, contra o resto do mesmo ano
lin = []
for ano in ANOS:
    for cc in PAISES:
        m = (d_lex["cc"] == cc) & (d_lex["year"] == ano)
        if m.sum() < 5:
            continue
        for r, (w, z, f) in enumerate(top_termos(m, (d_lex["year"] == ano) & ~m, 8, min_cont=6), 1):
            lin.append({"cc": cc, "year": ano, "rank": r, "termo": w, "z": z, "freq_por_mil": f})
lex_pais_ano = pd.DataFrame(lin)
lex_pais_ano.to_csv(PASTA_LEX / "palavras_pais_ano.csv", index=False)
tab = (lex_pais_ano[lex_pais_ano["rank"] <= 3].groupby(["cc", "year"])["termo"].apply(", ".join)
       .unstack("year").reindex(PAISES).rename(index=NOMES))
print("\nTrês palavras mais distintivas por país e ano (tabela completa em lexico/palavras_pais_ano.csv):")
with pd.option_context("display.max_colwidth", 40, "display.width", 250):
    display(tab)

# (d) palavras que sobem ou descem entre 2017 e 2025 (todos os países juntos)
Y = np.stack([conta(d_lex["year"] == a) for a in ANOS])
R = 1000 * Y / Y.sum(1, keepdims=True)
x = ANOS - ANOS.mean()
slope = (R - R.mean(0)).T @ x / (x @ x)
rel = slope / np.maximum(R.mean(0), 1e-9)
ok = Y.sum(0) >= 40
sobem = [k for k in np.argsort(-np.where(ok, rel, -np.inf))[:8]]
descem = [k for k in np.argsort(np.where(ok, rel, np.inf))[:8]]
print("\nPalavras que mais crescem (variação relativa por ano):", ", ".join(f"{VOC[k]} ({rel[k]:+.0%})" for k in sobem))
print("Palavras que mais caem:", ", ".join(f"{VOC[k]} ({rel[k]:+.0%})" for k in descem))
fig, ax = plt.subplots(figsize=(7.5, 3.6))
for k in sobem[:5]:
    ax.plot(ANOS, R[:, k], marker="o", color=COR["s1"], alpha=0.8); ax.annotate(VOC[k], (ANOS[-1], R[-1, k]), xytext=(5, 0), textcoords="offset points", fontsize=8)
for k in descem[:5]:
    ax.plot(ANOS, R[:, k], marker="o", color=COR["s2"], alpha=0.8); ax.annotate(VOC[k], (ANOS[-1], R[-1, k]), xytext=(5, 0), textcoords="offset points", fontsize=8)
ax.set_xticks(ANOS); ax.set_ylabel("ocorrências por mil palavras")
ax.set_title("Palavras em ascensão (azul) e em queda (laranja)", loc="left")
fig.savefig(PASTA_LEX / "palavras_tendencia.png"); plt.show()

## 12.3 Similaridade lexical entre países

Mesmo desenho de $S_t$, mas sobre **distribuições de palavras** em vez de clusters: para cada par de países e ano, $1-JS$ entre as frequências de palavras e cosseno entre vetores TF-IDF (com log nas contagens). Não depende de $k$ nem do UMAP, então serve de teste independente de H1 e H2. O IC vem de bootstrap das músicas dentro de cada célula (igual à seção 5) e o **nulo por permutação** mostra a similaridade esperada se o país não importasse, o que também corrige o viés de amostra pequena.

In [ ]:
# 12.3 — similaridade lexical (a mesma função serve para tópicos, na seção 12.9)
PARES = list(itertools.combinations(PAISES, 2))
_I = np.array([PAISES.index(a) for a, b in PARES]); _J = np.array([PAISES.index(b) for a, b in PARES])

def _metricas(C, peso, sublinear):
    tot = C.sum(1, keepdims=True)
    P = C / np.where(tot > 0, tot, 1)
    js = 1 - js_div(P[_I], P[_J])
    W = (np.log1p(C) if sublinear else C) * (1 if peso is None else peso)
    Wn = W / np.maximum(np.linalg.norm(W, axis=1, keepdims=True), 1e-12)
    cos = (Wn[_I] * Wn[_J]).sum(1)
    vazio = (tot.ravel()[_I] == 0) | (tot.ravel()[_J] == 0)
    js[vazio] = np.nan; cos[vazio] = np.nan
    return js, cos

def lex_series(M, d, peso=None, sublinear=True, n_boot=N_BOOT_LEX, n_perm=N_PERM_LEX, seed=SEED):
    # M: matriz por faixa (contagens de palavras ou mistura de tópicos), linhas alinhadas a d["ix"].
    rng = np.random.default_rng(seed)
    T, P, nP = len(ANOS), len(PARES), len(PAISES)
    obs = {"js": np.full((T, P), np.nan), "cos": np.full((T, P), np.nan)}
    boot = {"js": np.full((n_boot, T, P), np.nan), "cos": np.full((n_boot, T, P), np.nan)}
    nulo = {"js": np.full(T, np.nan), "cos": np.full(T, np.nan)}
    M = sp.csr_matrix(M) if not sp.issparse(M) else M
    for t, ano in enumerate(ANOS):
        g = d[d["year"] == ano]
        Ms = M[g["ix"].to_numpy()]
        cc = g["cc"].map({c: i for i, c in enumerate(PAISES)}).to_numpy()
        def soma(w_por_linha, rotulos):
            G = sp.csr_matrix((w_por_linha, (rotulos, np.arange(len(rotulos)))), shape=(nP, len(rotulos)))
            return (G @ Ms).toarray()
        js, cs = _metricas(soma(np.ones(len(cc)), cc), peso, sublinear)
        obs["js"][t], obs["cos"][t] = js, cs
        idx = [np.where(cc == c)[0] for c in range(nP)]
        for b in range(n_boot):
            w = np.zeros(len(cc))
            for c in range(nP):
                if len(idx[c]):
                    w[idx[c]] = rng.multinomial(len(idx[c]), np.full(len(idx[c]), 1 / len(idx[c])))
            js, cs = _metricas(soma(w, cc), peso, sublinear)
            boot["js"][b, t], boot["cos"][b, t] = js, cs
        nj, nc = [], []
        for _ in range(n_perm):
            js, cs = _metricas(soma(np.ones(len(cc)), rng.permutation(cc)), peso, sublinear)
            nj.append(np.nanmean(js)); nc.append(np.nanmean(cs))
        nulo["js"][t], nulo["cos"][t] = np.mean(nj), np.mean(nc)
    return {"obs": obs, "boot": boot, "nulo": nulo}

def mascara_lex(grupo=None, h2=False):
    if h2:     # um país de GRUPO_H2 contra qualquer outro
        return np.array([(a in GRUPO_H2) != (b in GRUPO_H2) for a, b in PARES])
    if grupo is None:
        return np.ones(len(PARES), bool)
    return np.array([a in grupo and b in grupo for a, b in PARES])

def lex_resumo(res, mask):
    # série anual (média dos pares da máscara), IC95% por bootstrap e tendência (igual ao resumo() da seção 5)
    out, tend = {"ano": ANOS}, {}
    for met in ["js", "cos"]:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            o = np.nanmean(res["obs"][met][:, mask], axis=1)
            b = np.nanmean(res["boot"][met][:, :, mask], axis=2)
        v = ~np.isnan(o)
        xv = ANOS[v].astype(float); xc = xv - xv.mean()
        q_lo, q_hi = np.nanpercentile(b, [2.5, 97.5], axis=0)
        out[met] = o; out[f"{met}_lo"] = o - (np.nanmean(b, 0) - q_lo); out[f"{met}_hi"] = o + (q_hi - np.nanmean(b, 0))
        inc = (o[v] - o[v].mean()) @ xc / (xc @ xc)
        inc_b = (b[:, v] - b[:, v].mean(1, keepdims=True)) @ xc / (xc @ xc)
        lo = inc - (inc_b.mean() - np.percentile(inc_b, 2.5)); hi = inc + (np.percentile(inc_b, 97.5) - inc_b.mean())
        rho, p_rho = spearmanr(xv, o[v]); p_ols = linregress(xv, o[v]).pvalue
        sinal = "convergência" if lo > 0 else ("divergência" if hi < 0 else None)
        leitura = "sem tendência clara" if sinal is None else (sinal if p_ols < 0.05 else f"{sinal} fraca (só no bootstrap)")
        tend[met] = {"inclinacao_por_ano": inc, "ic95_lo": lo, "ic95_hi": hi, "p_regressao": p_ols,
                     "spearman_rho": rho, "spearman_p": p_rho, "leitura": leitura}
    return pd.DataFrame(out), tend

def plota_lex(series, titulo, arquivo, nulo=None, rot=("1 − JS (distribuição de palavras)", "Cosseno TF-IDF")):
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.9))
    for ax, met, nome in [(axes[0], "js", rot[0]), (axes[1], "cos", rot[1])]:
        for r, df, cor in series:
            ax.fill_between(df["ano"], df[f"{met}_lo"], df[f"{met}_hi"], color=cor, alpha=0.15, linewidth=0)
            ax.plot(df["ano"], df[met], color=cor, marker="o", label=r)
        if nulo is not None:
            ax.plot(ANOS, nulo[met], color=COR["neutro"], linestyle="--", label="esperado se o país não importasse")
        ax.set_xticks(ANOS); ax.set_title(nome, loc="left")
    axes[0].legend(loc="best", fontsize=9)
    fig.suptitle(titulo, x=0.01, ha="left", fontweight="bold", y=1.02)
    fig.tight_layout(); fig.savefig(PASTA_LEX / arquivo); plt.show()

res_w = lex_series(X, d_lex, peso=IDF, sublinear=True)
sw_glob, tw_glob = lex_resumo(res_w, mascara_lex())
sw_h1, tw_h1 = lex_resumo(res_w, mascara_lex(GRUPO_H1))
sw_h2, tw_h2 = lex_resumo(res_w, mascara_lex(h2=True))
plota_lex([("todos os pares", sw_glob, COR["s1"])], "Similaridade lexical global entre os 6 países",
          "lex_St.png", nulo=res_w["nulo"])
plota_lex([("H1: EUA–BR–MX–ES", sw_h1, COR["s1"]), ("H2: Japão × demais", sw_h2, COR["s2"])],
          "H1 e H2 sobre o vocabulário", "lex_H1_H2.png")
tab_lex = tabela_tendencias({"lexical global": tw_glob, "lexical H1": tw_h1, "lexical H2": tw_h2})
display(tab_lex)
print("Nulo por permutação (1−JS lexical):", dict(zip(ANOS, res_w["nulo"]["js"].round(3))))

# pares: média e tendência, lado a lado com a medida por clusters (seção 6)
lin = []
for (a, b), msk in zip(PARES, np.eye(len(PARES), dtype=bool)):
    _, t = lex_resumo(res_w, msk)
    lin.append({"a": a, "b": b, "media_js_lex": np.nanmean(res_w["obs"]["js"][:, msk]), "incl_js_lex": t["js"]["inclinacao_por_ano"],
                "leitura_lex": t["js"]["leitura"]})
lex_pares = pd.DataFrame(lin)
def _mat(col):
    Mx = pd.DataFrame(np.nan, index=PAISES, columns=PAISES)
    for r in lex_pares.itertuples():
        Mx.loc[r.a, r.b] = Mx.loc[r.b, r.a] = getattr(r, col)
    return Mx
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
im0 = plota_matriz(_mat("media_js_lex"), "Similaridade lexical média (1 − JS)", axes[0], SEQ); fig.colorbar(im0, ax=axes[0], shrink=0.8)
Mi = _mat("incl_js_lex"); lim = np.nanmax(np.abs(Mi.values))
im1 = plota_matriz(Mi, "Tendência por ano (azul = aproximando)", axes[1], DIV, vmin=-lim, vmax=lim, fmt="{:+.3f}"); fig.colorbar(im1, ax=axes[1], shrink=0.8)
fig.tight_layout(); fig.savefig(PASTA_LEX / "lex_pares.png"); plt.show()
if "tend_pares" in globals():
    cmp_ = lex_pares.merge(tend_pares[["a", "b", "media_js", "inclinacao_js"]], on=["a", "b"])
    r1 = spearmanr(cmp_["media_js_lex"], cmp_["media_js"]); r2 = spearmanr(cmp_["incl_js_lex"], cmp_["inclinacao_js"])
    print(f"Concordância com a medida por clusters (15 pares): Spearman nível = {r1[0]:.2f} (p={r1[1]:.3f}) | tendência = {r2[0]:.2f} (p={r2[1]:.3f})")
lex_pares.to_csv(PASTA_LEX / "lex_pares.csv", index=False)

## 12.4 Vizinhos semânticos e frases parecidas

Aqui buscamos **exemplos**: pares de músicas de países diferentes, no mesmo ano, mais próximas no espaço de embeddings (que é multilíngue, então vale entre idiomas). Excluímos a mesma música com versões diferentes. Depois, para os pares mais próximos, comparamos **linha a linha** para achar as frases mais parecidas. É uma leitura qualitativa que ilustra o que a métrica agregada significa, e ajuda a separar "parecido por tema" de "parecido por ser a mesma fórmula".

O trecho de frases usa o modelo de embeddings (já carregado pela seção 2); se ele não estiver disponível, só a tabela de pares é gerada.

In [ ]:
# 12.4 — pares de músicas próximas entre países e frases semelhantes
N_PARES_FRASES = 12          # quantos pares recebem a comparação linha a linha
LINHAS_POR_FAIXA = 80

def _artista1(a):
    return re.split(r"\s*(?:\||,|&)\s*", str(a))[0].strip().lower()

rows = []
for ano, g in dados.groupby("year"):
    idx = g["track_id"].map(POS).to_numpy()
    S = E_orig[idx] @ E_orig[idx].T
    cc = g["cc"].to_numpy(); ch = g["chave"].to_numpy()
    ok = (cc[:, None] != cc[None, :]) & (ch[:, None] != ch[None, :])
    iu = np.triu_indices(len(g), 1)
    s, ok = S[iu], ok[iu]
    s = np.where(ok, s, -1)
    top = np.argsort(-s)[:15]
    for t in top:
        i, j = iu[0][t], iu[1][t]
        rows.append({"year": ano, "cos": float(s[t]),
                     "cc_a": cc[i], "titulo_a": g["title"].iloc[i], "artista_a": g["artists"].iloc[i], "id_a": g["track_id"].iloc[i],
                     "cc_b": cc[j], "titulo_b": g["title"].iloc[j], "artista_b": g["artists"].iloc[j], "id_b": g["track_id"].iloc[j],
                     "mesmo_artista": _artista1(g["artists"].iloc[i]) == _artista1(g["artists"].iloc[j])})
viz = pd.DataFrame(rows).sort_values("cos", ascending=False).drop_duplicates(["id_a", "id_b"]).reset_index(drop=True)
viz_sel = viz[~viz["mesmo_artista"]].head(N_PARES_FRASES)
print("Pares de países que mais aparecem entre os vizinhos mais próximos:")
_n = [NOMES[c] for c in PAISES]
_ct = lambda a, b: pd.crosstab(viz[a].map(NOMES), viz[b].map(NOMES)).reindex(index=_n, columns=_n, fill_value=0)
display(_ct("cc_a", "cc_b") + _ct("cc_b", "cc_a"))
with pd.option_context("display.max_colwidth", 40, "display.width", 250):
    display(viz_sel[["year", "cos", "cc_a", "titulo_a", "artista_a", "cc_b", "titulo_b", "artista_b"]].round(3))
viz.to_csv(PASTA_LEX / "vizinhos_semanticos.csv", index=False)

def _linhas(tid):
    ls = [l.strip() for l in str(faixas.loc[tid, "letra_limpa"]).split("\n") if len(l.strip()) >= 12]
    return list(dict.fromkeys(ls))[:LINHAS_POR_FAIXA]

def frases_parecidas(tid_a, tid_b, k=3):
    la, lb = _linhas(tid_a), _linhas(tid_b)
    if not la or not lb:
        return []
    m = carrega_modelo(); pref = "passage: " if "e5" in MODELO.lower() else ""
    Va = m.encode([pref + l for l in la], normalize_embeddings=True, batch_size=BATCH, convert_to_numpy=True)
    Vb = m.encode([pref + l for l in lb], normalize_embeddings=True, batch_size=BATCH, convert_to_numpy=True)
    S = Va @ Vb.T
    out = []
    for _ in range(k):
        i, j = np.unravel_index(np.argmax(S), S.shape)
        out.append((float(S[i, j]), la[i], lb[j]))
        S[i, :] = -1; S[:, j] = -1
    return out

try:
    saida = []
    for r in viz_sel.itertuples():
        for s, la_, lb_ in frases_parecidas(r.id_a, r.id_b):
            saida.append({"cos_linha": s, "par": f"{r.titulo_a} ({NOMES[r.cc_a]}) × {r.titulo_b} ({NOMES[r.cc_b]})", "linha_a": la_, "linha_b": lb_})
    frases = pd.DataFrame(saida)
    with pd.option_context("display.max_colwidth", 70, "display.width", 250):
        display(frases.sort_values("cos_linha", ascending=False).head(30).round(3))
    frases.to_csv(PASTA_LEX / "frases_parecidas.csv", index=False)
except Exception as e:
    print("comparação linha a linha não executada:", repr(e)[:150])

## 12.5 Frases (trigramas) compartilhadas entre países

Se há circulação de um repertório global, esperamos **fórmulas verbais** repetidas em países diferentes (por exemplo, "you and me", "all night long"). Contamos trigramas de palavras (com ao menos duas palavras de conteúdo) que aparecem em músicas de **3 ou mais países no mesmo ano**, usando só as músicas *exclusivas* (para não contar a mesma letra duas vezes). O indicador é a fração desses trigramas entre todos os trigramas presentes no ano. Depende do tamanho da amostra, então a leitura é descritiva.

In [ ]:
# 12.5 — trigramas compartilhados
tv = CountVectorizer(ngram_range=(3, 3), binary=True, min_df=3, lowercase=True, preprocessor=_prep, token_pattern=r"(?u)\b[a-zA-Z]+\b")
T3 = tv.fit_transform(fx[FONTE_TEXTO]).tocsr()
V3 = np.array(tv.get_feature_names_out())
util = np.array([sum(w not in STOP for w in g.split()) >= 2 and len(set(g.split())) >= 2 for g in V3])
T3, V3 = T3[:, util], V3[util]
print(f"{len(V3)} trigramas candidatos (≥2 palavras de conteúdo, em ≥3 músicas)")

excl_lex = d_lex[~d_lex["compartilhada"]]
lin, acum = [], pd.Series(0, index=V3, dtype=float)
for ano in ANOS:
    g = excl_lex[excl_lex["year"] == ano]
    pres = np.stack([np.asarray(T3[g.loc[g["cc"] == c, "ix"].to_numpy()].sum(0)).ravel() > 0
                     if (g["cc"] == c).any() else np.zeros(len(V3), bool) for c in PAISES])
    n_pais = pres.sum(0)
    tipos = (n_pais >= 1).sum()
    comp3 = n_pais >= 3
    acum[comp3] += n_pais[comp3]
    lin.append({"year": ano, "trigramas_no_ano": int(tipos), "em_3+_paises": int(comp3.sum()),
                "fracao": comp3.sum() / max(tipos, 1), "em_2_paises": int((n_pais == 2).sum())})
tri = pd.DataFrame(lin)
display(tri.round(3))
print("Trigramas mais espalhados (nº de países-ano em que aparecem em 3+ países):")
print(", ".join(f"{k} ({int(v)})" for k, v in acum.sort_values(ascending=False).head(20).items() if v > 0))
fig, ax = plt.subplots(figsize=(6, 2.8))
ax.plot(tri["year"], 100 * tri["fracao"], color=COR["s1"], marker="o")
ax.set_xticks(tri["year"]); ax.set_ylabel("% dos trigramas")
ax.set_title("Trigramas presentes em 3+ países no mesmo ano", loc="left")
fig.savefig(PASTA_LEX / "trigramas_compartilhados.png"); plt.show()
tri.to_csv(PASTA_LEX / "trigramas_compartilhados.csv", index=False)

## 12.6 Homofilia: músicas de um país têm vizinhos do mesmo país?

*Homofilia* (McPherson, Smith-Lovin e Cook, 2001) é a tendência de laços ocorrerem entre semelhantes. Aqui o "laço" é a vizinhança semântica: para cada música, olhamos seus $k$ vizinhos mais próximos no mesmo ano e medimos a fração que é do mesmo país. O índice normalizado é $H=(obs-esp)/(1-esp)$, em que $esp$ é o valor esperado ao acaso: $H=0$ é mistura total e $H=1$ é separação total. **Se os países convergem, $H$ cai.** Vantagem sobre $S_t$: não usa clusters, só distâncias, e a banda cinza (permutação dos rótulos de país) mostra o que o acaso produziria. Rodamos com os embeddings originais e com os da tradução.

In [ ]:
# 12.6 — homofilia
def homofilia(E, d, pos, k=K_VIZ, n_perm=200, seed=SEED):
    rng = np.random.default_rng(seed)
    lin = []
    for ano, g in d.groupby("year"):
        idx = g["track_id"].map(pos).to_numpy()
        S = E[idx] @ E[idx].T
        ch = g["chave"].to_numpy(); cc = g["cc"].to_numpy()
        S[ch[:, None] == ch[None, :]] = -np.inf               # ignora a própria música e outras versões dela
        kk = min(k, len(g) - 2)
        nn = np.argpartition(-S, kk, axis=1)[:, :kk]
        cont = pd.Series(cc).value_counts()
        n_c = pd.Series(cc).map(cont).to_numpy()
        esp = float(((n_c - 1) / (len(cc) - 1)).mean())
        h = lambda lab: ((lab[nn] == lab[:, None]).mean() - esp) / (1 - esp)
        nulo = [h(rng.permutation(cc)) for _ in range(n_perm)]
        lin.append({"year": ano, "H": h(cc), "nulo_lo": np.percentile(nulo, 2.5), "nulo_hi": np.percentile(nulo, 97.5)})
    return pd.DataFrame(lin)

hom = {"original": homofilia(E_orig, d_lex, POS)}
E_trad = None
if FONTE_TEXTO == "letra_en":
    try:
        E_trad = gera_embeddings(fx["letra_en"], "traducao_en")
        hom["traduzida"] = homofilia(E_trad, d_lex, LPOS)
    except Exception as e:
        print("homofilia com tradução não executada:", repr(e)[:120])

def tend_simples(anos, y):
    anos, y = np.asarray(anos, float), np.asarray(y, float)
    v = ~np.isnan(y)
    if v.sum() < 3 or np.ptp(y[v]) == 0:
        return {"inclinacao_por_ano": np.nan, "p_regressao": np.nan, "spearman_rho": np.nan, "spearman_p": np.nan}
    r = linregress(anos[v], y[v]); rho, p = spearmanr(anos[v], y[v])
    return {"inclinacao_por_ano": r.slope, "p_regressao": r.pvalue, "spearman_rho": rho, "spearman_p": p}

fig, ax = plt.subplots(figsize=(7, 3.6))
for (nome, h), cor in zip(hom.items(), [COR["s1"], COR["s2"]]):
    ax.plot(h["year"], h["H"], marker="o", color=cor, label=nome)
ax.fill_between(h["year"], h["nulo_lo"], h["nulo_hi"], color=COR["neutro"], alpha=0.3, label="acaso (95%)")
ax.set_xticks(h["year"]); ax.set_ylabel("H (0 = mistura, 1 = separação)"); ax.legend(fontsize=9)
ax.set_title("Homofilia por país na vizinhança semântica", loc="left")
fig.savefig(PASTA_LEX / "homofilia.png"); plt.show()
tab_hom = pd.DataFrame({n: tend_simples(h["year"], h["H"]) for n, h in hom.items()}).T.round(4)
print("Tendência de H (inclinação negativa = países se misturando):"); display(tab_hom)

## 12.7 Previsibilidade do país a partir do texto

Teste de duas amostras por classificador (Lopez-Paz e Oquab, 2017): se um modelo simples consegue adivinhar o país de uma música só pelo texto, as distribuições são distinguíveis. Treinamos uma regressão logística **dentro de cada ano** com validação cruzada (5 partes, 3 repetições) e reportamos a acurácia balanceada. O acaso é 1/6 no problema de 6 países e 0,5 nos pares. **Se os países convergem, a acurácia cai.** Complementa $S_t$ porque usa a informação de todo o vetor, não só do cluster, e deixa claro quanto do sinal é idioma: compare original e traduzida.

In [ ]:
# 12.7 — previsibilidade do país
def acc_cv(Xm, y, reps=3, seed=SEED):
    y = np.asarray(y)
    if len(set(y)) < 2 or pd.Series(y).value_counts().min() < 5:
        return np.nan
    accs = []
    for r in range(reps):
        pred = np.empty(len(y), dtype=object)
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed + r).split(Xm, y):
            clf = LogisticRegression(max_iter=2000, C=1.0).fit(Xm[tr], y[tr])
            pred[te] = clf.predict(Xm[te])
        accs.append(balanced_accuracy_score(y, pred))
    return float(np.mean(accs))

def previsibilidade(E, d, pos):
    lin = []
    for ano, g in d.groupby("year"):
        Xm = E[g["track_id"].map(pos).to_numpy()]; y = g["cc"].to_numpy()
        h1 = [acc_cv(Xm[np.isin(y, [a, b])], y[np.isin(y, [a, b])]) for a, b in itertools.combinations(GRUPO_H1, 2)]
        h2 = [acc_cv(Xm[np.isin(y, [a, o])], y[np.isin(y, [a, o])]) for a in GRUPO_H2 for o in PAISES if o not in GRUPO_H2]
        lin.append({"year": ano, "global_6": acc_cv(Xm, y), "H1_pares": np.nanmean(h1), "H2_pares": np.nanmean(h2)})
    return pd.DataFrame(lin)

prev = {"original": previsibilidade(E_orig, d_lex, POS)}
if E_trad is not None:
    prev["traduzida"] = previsibilidade(E_trad, d_lex, LPOS)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharex=True)
for ax, col, ch_, nome in [(axes[0], "global_6", 1 / 6, "6 países (acaso = 0,17)"),
                           (axes[1], "H1_pares", 0.5, "pares do bloco H1 (acaso = 0,5)"),
                           (axes[2], "H2_pares", 0.5, "Japão × demais (acaso = 0,5)")]:
    for (n, p), cor in zip(prev.items(), [COR["s1"], COR["s2"]]):
        ax.plot(p["year"], p[col], marker="o", color=cor, label=n)
    ax.axhline(ch_, color=COR["neutro"], linestyle="--"); ax.set_title(nome, loc="left", fontsize=10); ax.set_xticks(p["year"])
    ax.tick_params(axis="x", labelsize=8)
axes[0].legend(fontsize=9); fig.suptitle("Quão bem o texto revela o país (acurácia balanceada)", x=0.01, ha="left", fontweight="bold", y=1.03)
fig.tight_layout(); fig.savefig(PASTA_LEX / "previsibilidade.png"); plt.show()
tab_prev = pd.concat({n: pd.DataFrame({c: tend_simples(p["year"], p[c]) for c in ["global_6", "H1_pares", "H2_pares"]}).T for n, p in prev.items()}).round(4)
print("Tendência da acurácia (inclinação negativa = países ficando menos distinguíveis):"); display(tab_prev)

## 12.8 Informação mútua país–cluster e diversidade interna

Quanto do cluster de uma música é "explicado" pelo seu país? A informação mútua normalizada (NMI) entre país e cluster, **calculada ano a ano**, resume isso: se cai, os clusters deixam de ser marcas de país, o que é a definição operacional de homogeneização entre países. Em paralelo, a **entropia de Shannon** da distribuição de clusters de cada país mede a **diversidade interna**: homogeneização global não é o mesmo que perda de diversidade dentro dos países, e a teoria distingue os dois.

In [ ]:
# 12.8 — MI país–cluster e diversidade interna
mi = pd.DataFrame([{"year": a, "NMI_pais_cluster": normalized_mutual_info_score(g["cc"], g["cluster"])}
                   for a, g in dados.groupby("year")])
ent = (dados.groupby(["cc", "year"])["cluster"].apply(lambda s: float(entropy(s.value_counts(normalize=True), base=2)))
       .unstack("cc").reindex(columns=PAISES))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(mi["year"], mi["NMI_pais_cluster"], marker="o", color=COR["s1"]); axes[0].set_xticks(mi["year"])
axes[0].set_title("NMI entre país e cluster, por ano", loc="left")
for cc in PAISES:
    axes[1].plot(ent.index, ent[cc], marker="o", color=COR_PAIS[cc], label=NOMES[cc])
axes[1].set_xticks(ent.index); axes[1].set_title("Diversidade interna (entropia dos clusters, bits)", loc="left"); axes[1].legend(fontsize=8, ncol=2)
fig.tight_layout(); fig.savefig(PASTA_LEX / "mi_diversidade.png"); plt.show()
print("Tendência do NMI país–cluster:", {k: round(v, 4) for k, v in tend_simples(mi["year"], mi["NMI_pais_cluster"]).items()})
print("Tendência da diversidade interna (inclinação em bits/ano):")
display(pd.DataFrame({NOMES[c]: tend_simples(ent.index, ent[c]) for c in PAISES}).T.round(4))

## 12.9 Tópicos (LDA): convergência temática sem k-means

O LDA (Blei et al., 2003) descreve cada música como uma mistura de tópicos, e a análise cultural por tópicos é um recurso clássico da sociologia da cultura (DiMaggio, Nag e Blei, 2013). Aqui serve de **segunda definição de "tema"**, independente do UMAP e do K-Means: se H1 e H2 aparecem também sobre tópicos, o resultado não é artefato da clusterização. Usamos a mesma função de similaridade, agora sobre as misturas de tópicos.

In [ ]:
# 12.9 — LDA e similaridade temática
lda = LatentDirichletAllocation(n_components=N_TOPICOS, random_state=SEED, learning_method="batch", max_iter=40)
theta = lda.fit_transform(X)
print("Tópicos (palavras de maior peso):")
for t in range(N_TOPICOS):
    print(f"  T{t}: " + ", ".join(VOC[np.argsort(-lda.components_[t])[:10]]))

prev_top = pd.DataFrame(theta[d_lex["ix"].to_numpy()]).groupby(d_lex["cc"].to_numpy()).mean().reindex(PAISES)
fig, ax = plt.subplots(figsize=(1.0 + 0.55 * N_TOPICOS, 3.4))
im = plota_matriz(prev_top.rename(columns=lambda c: f"T{c}"), "Peso médio de cada tópico por país", ax, SEQ, vmin=0, fmt="{:.2f}")
fig.colorbar(im, ax=ax, shrink=0.8); fig.savefig(PASTA_LEX / "topicos_por_pais.png"); plt.show()

res_t = lex_series(theta, d_lex, peso=None, sublinear=False)
st_glob, tt_glob = lex_resumo(res_t, mascara_lex())
st_h1, tt_h1 = lex_resumo(res_t, mascara_lex(GRUPO_H1))
st_h2, tt_h2 = lex_resumo(res_t, mascara_lex(h2=True))
plota_lex([("H1: EUA–BR–MX–ES", st_h1, COR["s1"]), ("H2: Japão × demais", st_h2, COR["s2"])],
          "H1 e H2 sobre a mistura de tópicos (LDA)", "lda_H1_H2.png", rot=("1 − JS (mistura de tópicos)", "Cosseno entre misturas"))
tab_lda = tabela_tendencias({"tópicos global": tt_glob, "tópicos H1": tt_h1, "tópicos H2": tt_h2}); display(tab_lda)

## 12.10 Tom emocional e estilo: sentimento, repetição e compressibilidade

Convergência pode ser de **estilo** e não só de tema. Três indicadores por música, agregados por país e ano:

- **Sentimento (VADER)**, média por linha do texto em inglês. Segue a ideia de medir o "bem-estar" expresso em canções (Dodds e Danforth, 2010), com um léxico de sentimento (Hutto e Gilbert, 2014). Requer `pip install vaderSentiment`.
- **Repetição de linhas**: 1 − (linhas distintas / linhas), proxy de refrão e estrutura repetitiva.
- **Taxa de compressão gzip**: tamanho comprimido / tamanho original; quanto menor, mais repetitivo o texto (aproxima a complexidade).

Em cada um, olhamos (a) a tendência da **média geral** e (b) a **dispersão entre países** por ano: dispersão em queda = países ficando mais parecidos naquela dimensão.

In [ ]:
# 12.10 — sentimento, repetição e compressibilidade
def repete(t):
    ls = [l.strip().lower() for l in str(t).split("\n") if l.strip()]
    return 1 - len(set(ls)) / len(ls) if ls else np.nan

def comprime(t):
    b = str(t).encode("utf-8")
    return len(gzip.compress(b)) / max(len(b), 1)

est = pd.DataFrame({"repeticao_linhas": fx[FONTE_TEXTO].map(repete), "razao_gzip": fx[FONTE_TEXTO].map(comprime)}, index=fx.index)
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _sia = SentimentIntensityAnalyzer()
    est["sentimento"] = fx[FONTE_TEXTO].map(lambda t: np.mean([_sia.polarity_scores(l)["compound"] for l in str(t).split("\n") if l.strip()] or [np.nan]))
except ImportError:
    print("vaderSentiment não instalado: pulando o sentimento (pip install vaderSentiment)")

dl = d_lex.join(est, on="track_id")
indicadores = [c for c in ["sentimento", "repeticao_linhas", "razao_gzip"] if c in est.columns]
fig, axes = plt.subplots(2, len(indicadores), figsize=(4.2 * len(indicadores), 6), squeeze=False)
lin = []
for j, ind in enumerate(indicadores):
    mp = dl.groupby(["year", "cc"])[ind].mean().unstack("cc").reindex(columns=PAISES)
    disp = mp.std(axis=1)
    for cc in PAISES:
        axes[0, j].plot(mp.index, mp[cc], marker="o", color=COR_PAIS[cc], label=NOMES[cc])
    axes[0, j].set_title(f"{ind}: média por país", loc="left", fontsize=10); axes[0, j].set_xticks(mp.index); axes[0, j].tick_params(axis="x", labelsize=7)
    axes[1, j].plot(disp.index, disp.values, marker="o", color=COR["s1"]); axes[1, j].set_xticks(disp.index); axes[1, j].tick_params(axis="x", labelsize=7)
    axes[1, j].set_title(f"{ind}: dispersão entre países", loc="left", fontsize=10)
    lin.append({"indicador": ind, "nivel_medio": tend_simples(mp.index, mp.mean(axis=1))["inclinacao_por_ano"],
                "p_nivel": tend_simples(mp.index, mp.mean(axis=1))["p_regressao"],
                "incl_dispersao": tend_simples(disp.index, disp.values)["inclinacao_por_ano"],
                "p_dispersao": tend_simples(disp.index, disp.values)["p_regressao"]})
axes[0, 0].legend(fontsize=7, ncol=2); fig.tight_layout(); fig.savefig(PASTA_LEX / "estilo_sentimento.png"); plt.show()
tab_estilo = pd.DataFrame(lin).round(4)
print("Inclinação por ano do nível médio e da dispersão entre países (dispersão em queda = convergência):"); display(tab_estilo)

## 12.11 Resumo e arquivos salvos

Para cada medida, a pergunta é a mesma: a similaridade entre países **sobe** (convergência) ou **desce** (divergência) com o tempo? Só chamamos de tendência quando o IC exclui zero e a regressão concorda, como na seção 5. Concordância entre métodos independentes (clusters, vocabulário, tópicos, homofilia, classificador) é o que sustenta a conclusão; discordância deve ser reportada.

In [ ]:
# 12.11 — resumo e arquivos
todas_lex = pd.concat([tab_lex, tab_lda], ignore_index=True)
todas_lex.to_csv(PASTA_LEX / "tendencias_lexicas.csv", index=False)
tab_hom.to_csv(PASTA_LEX / "homofilia_tendencia.csv"); tab_prev.to_csv(PASTA_LEX / "previsibilidade_tendencia.csv")
tab_estilo.to_csv(PASTA_LEX / "estilo_tendencia.csv", index=False)
lex_cluster.to_csv(PASTA_LEX / "palavras_cluster.csv", index=False); lex_pais.to_csv(PASTA_LEX / "palavras_pais.csv", index=False)

def _f(t):
    return f"{t['inclinacao_por_ano']:+.4f}/ano [IC95% {t['ic95_lo']:+.4f}, {t['ic95_hi']:+.4f}] p_reg={t['p_regressao']:.3f} -> {t['leitura']}"
print("Similaridade lexical global (1−JS) .. ", _f(tw_glob["js"]))
print("Lexical H1 (1−JS) ................... ", _f(tw_h1["js"]))
print("Lexical H2 (1−JS) ................... ", _f(tw_h2["js"]))
print("Tópicos H1 (1−JS) ................... ", _f(tt_h1["js"]))
print("Tópicos H2 (1−JS) ................... ", _f(tt_h2["js"]))
for n, r in tab_hom.iterrows():
    print(f"Homofilia H ({n}) ........ {r['inclinacao_por_ano']:+.4f}/ano (p={r['p_regressao']:.3f})")
print(f"\nArquivos em {PASTA_LEX}")